<a href="https://colab.research.google.com/github/Yamato603/colab-Pix2Pix/blob/main/Pix2Pix_Macular_Hole.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix

Cloning into 'pytorch-CycleGAN-and-pix2pix'...
remote: Enumerating objects: 2619, done.
remote: Total 2619 (delta 0), reused 0 (delta 0), pack-reused 2619 (from 1)
Receiving objects: 100% (2619/2619), 8.24 MiB | 26.27 MiB/s, done.
Resolving deltas: 100% (1654/1654), done.


In [2]:
import os
os.chdir('pytorch-CycleGAN-and-pix2pix/')

In [3]:
# 必要な3つの追加ツール（dominate, visdom, wandb）を直接インストールする
!pip install dominate visdom wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for visdom: filename=visdom-0.2.4-py3-none-any.whl size=1408195 sha256=5185ea89dcc54d1a4b95d5861a43e7bf9d9379e356c627c9ce66e66e985c5fb5
  Stored in directory: /root/.cache/pip/wheels/37/6c/38/64eeaa310e325aacda723e6df1f79ab5e9f31ba195264e04a8
Successfully built visdom


In [4]:
# スクリプトの先頭にインポート文を挿入して修正
!sed -i '1i from pathlib import Path' /content/pytorch-CycleGAN-and-pix2pix/datasets/combine_A_and_B.py

print("スクリプトの修正が完了しました！")

スクリプトの修正が完了しました！


In [5]:
import os
import glob
import random
from PIL import Image
from google.colab import drive

# =========================================================
# Step 1: Google Driveの連携とZIPファイルの解凍
# =========================================================
print("--- Step 1: Google Driveを連携します ---")
drive.mount('/content/drive')

# ZIPファイルのパス（ご自身のDriveの保存場所）
zip_path_post = '/content/drive/MyDrive/Postoperative_Saga_Cross-Section_Images.zip'
zip_path_pre = '/content/drive/MyDrive/Preoperative_Saga_Cross-Section_Images.zip'

# 作業用・保存用フォルダの定義
raw_base_dir = '/content/pytorch-CycleGAN-and-pix2pix/datasets/oct_raw'
out_dir = '/content/pytorch-CycleGAN-and-pix2pix/datasets/oct_pix2pix'

# 過去の古いフォルダがあればリセットして綺麗にする
os.system(f'rm -rf "{raw_base_dir}"')
os.system(f'rm -rf "{out_dir}"')
os.makedirs(raw_base_dir, exist_ok=True)

print("--- Step 2: ZIPファイルを解凍しています ---")
os.system(f'unzip -q -o "{zip_path_pre}" -d "{raw_base_dir}"')
os.system(f'unzip -q -o "{zip_path_post}" -d "{raw_base_dir}"')
print("解凍が完了しました！\n")


# =========================================================
# Step 2: 画像のペアリング・リサイズ・結合処理
# =========================================================
print("--- Step 3: 画像ペアの探索とPix2Pix用結合を開始します ---")

# ★マークがついた「特に病状が表れている画像」だけを使う場合は True に変更してください
USE_ONLY_STAR = False

# 目標画像サイズ (元画像 690x345 と同じ 2:1 比率)
target_w = 512
target_h = 256

# 解凍先の親フォルダを探す
pre_dir = os.path.join(raw_base_dir, 'Preoperative_Saga_Cross-Section_Images')
post_dir = os.path.join(raw_base_dir, 'Postoperative_Saga_Cross-Section_Images')

# サブフォルダ（1, 2...）以下のすべての画像ファイルを取得
pre_images = glob.glob(os.path.join(pre_dir, "**", "*.*"), recursive=True)

pairs = []
for pre_path in pre_images:
    # ★修正点: .bmp と .bpm も読み込み対象に追加しました
    if not pre_path.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.bpm')):
        continue

    # 患者番号（フォルダ名）とファイル名を取得
    patient_dir = os.path.basename(os.path.dirname(pre_path))
    filename = os.path.basename(pre_path)

    # ★縛りの判定
    if USE_ONLY_STAR and '★' not in filename:
        continue

    # 対応する術後(Postoperative)画像のパスを構築
    post_path = os.path.join(post_dir, patient_dir, filename)

    # 片方にしか★がついていない場合などの表記揺れ対策
    if not os.path.exists(post_path):
        if '★' in filename:
            post_path_alt = os.path.join(post_dir, patient_dir, filename.replace('★', ''))
        else:
            post_path_alt = os.path.join(post_dir, patient_dir, '★' + filename)

        if os.path.exists(post_path_alt):
            post_path = post_path_alt
        else:
            print(f"[スキップ] 対応する術後画像が見つかりません: 患者[{patient_dir}] / {filename}")
            continue

    pairs.append((pre_path, post_path))

# ランダムにシャッフルしてデータ分割 (Train 80%, Val 10%, Test 10%)
random.seed(42)
random.shuffle(pairs)

total = len(pairs)
if total == 0:
    print("【エラー】有効な画像ペアが見つかりませんでした。ZIPファイル内の構造や画像形式を確認してください。")
else:
    train_split = int(total * 0.8)
    val_split = int(total * 0.9)

    splits = {
        'train': pairs[:train_split],
        'val': pairs[train_split:val_split],
        'test': pairs[val_split:]
    }

    print(f"合計 {total} ペアの有効データを見つけました。結合処理を実行します...")

    count = 0
    for split_name, pair_list in splits.items():
        split_dir = os.path.join(out_dir, split_name)
        os.makedirs(split_dir, exist_ok=True)

        for pre_path, post_path in pair_list:
            img_a = Image.open(pre_path).convert('RGB')
            img_b = Image.open(post_path).convert('RGB')

            # 高画質リサイズ (512x256)
            img_a = img_a.resize((target_w, target_h), Image.Resampling.LANCZOS)
            img_b = img_b.resize((target_w, target_h), Image.Resampling.LANCZOS)

            # 左右に連結 (1024x256)
            img_ab = Image.new('RGB', (target_w * 2, target_h))
            img_ab.paste(img_a, (0, 0))
            img_ab.paste(img_b, (target_w, 0))

            # AI学習用に安全な連番（.jpg）で一元化して保存します
            out_filename = f"{count:05d}.jpg"
            img_ab.save(os.path.join(split_dir, out_filename))
            count += 1

        print(f" ✔ 【{split_name}】: {len(pair_list)} ペアの処理と保存完了")

    print("\n=========================================================")
    print(" すべての準備が完了しました！")
    print(f" 保存先: {out_dir}")
    print("=========================================================")

--- Step 1: Google Driveを連携します ---
Mounted at /content/drive
--- Step 2: ZIPファイルを解凍しています ---
解凍が完了しました！

--- Step 3: 画像ペアの探索とPix2Pix用結合を開始します ---
合計 200 ペアの有効データを見つけました。結合処理を実行します...
 ✔ 【train】: 160 ペアの処理と保存完了
 ✔ 【val】: 20 ペアの処理と保存完了
 ✔ 【test】: 20 ペアの処理と保存完了

 すべての準備が完了しました！
 保存先: /content/pytorch-CycleGAN-and-pix2pix/datasets/oct_pix2pix


In [6]:
%cd /content/pytorch-CycleGAN-and-pix2pix

!python train.py \
  --dataroot ./datasets/oct_pix2pix \
  --name oct_macular_hole \
  --model pix2pix \
  --direction AtoB \
  --preprocess none \
  --batch_size 1 \
  --input_nc 1 \
  --output_nc 1 \
  --n_epochs 50 \
  --n_epochs_decay 50

/content
----------------- Options ---------------
               batch_size: 1                             
                    beta1: 0.5                           
          checkpoints_dir: ./checkpoints                 
           continue_train: False                         
                crop_size: 256                           
                 dataroot: ./datasets/oct_pix2pix        	[default: None]
             dataset_mode: aligned                       
                direction: AtoB                          
             display_freq: 400                           
          display_winsize: 256                           
                    epoch: latest                        
              epoch_count: 1                             
                 gan_mode: vanilla                       
                init_gain: 0.02                          
                init_type: normal                        
                 input_nc: 1                             	[defa

In [9]:
# 作業フォルダへ移動
%cd /content/pytorch-CycleGAN-and-pix2pix

!python test.py \
  --dataroot ./datasets/oct_pix2pix \
  --name oct_macular_hole \
  --model pix2pix \
  --direction AtoB \
  --preprocess none \
  --input_nc 1 \
  --output_nc 1 \
  --results_dir ./results/

/content
----------------- Options ---------------
             aspect_ratio: 1.0                           
               batch_size: 1                             
          checkpoints_dir: ./checkpoints                 
                crop_size: 256                           
                 dataroot: ./datasets/oct_pix2pix        	[default: None]
             dataset_mode: aligned                       
                direction: AtoB                          
          display_winsize: 256                           
                    epoch: latest                        
                     eval: False                         
                init_gain: 0.02                          
                init_type: normal                        
                 input_nc: 1                             	[default: 3]
                  isTrain: False                         	[default: None]
                load_iter: 0                             	[default: 0]
                load_

In [11]:
# 1. Google Drive内に保存用フォルダを作成（既にある場合は自動でスキップされます）
!mkdir -p /content/drive/MyDrive/OCT_Weights

# 2. Pix2Pixのチェックポイントフォルダをご自身のGoogle Driveへ丸ごとコピー
!cp -r /content/pytorch-CycleGAN-and-pix2pix/checkpoints/oct_macular_hole /content/drive/MyDrive/OCT_Weights/

print("Google Driveへの重みのバックアップが完了しました！")


Google Driveへの重みのバックアップが完了しました！
